# 構造化出力とエージェンティックワークフロー


## Structured Outputs（構造化出力）


In [ ]:
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import List


class Person(BaseModel):
    name: str
    age: int | None = Field(description="年齢、不明な場合はNone")


class PersonList(BaseModel):
    people: List[Person]


client = OpenAI()
response = client.chat.completions.parse(
    model="gpt-5-nano",
    messages=[
        {
            "role": "developer",
            "content": "人物を抽出してください。",
        },
        {
            "role": "user",
            "content": "昔々あるところにおじいさん(70)とおばあさん(年齢不詳)がいました。",
        },
    ],
    reasoning_effort="minimal",
    response_format=PersonList,
)

parsed_result = response.choices[0].message.parsed
print(type(parsed_result))
print(parsed_result)

## with_structured_output


In [ ]:
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model


class Recipe(BaseModel):
    ingredients: list[str] = Field(description="ingredients of the dish")
    steps: list[str] = Field(description="steps to make the dish")


model = init_chat_model(
    model="gpt-5-nano",
    model_provider="openai",
    reasoning_effort="minimal",
)
structured_model = model.with_structured_output(Recipe)
result = structured_model.invoke("カレーのレシピを教えて")

print(type(result))
print(result)

# Tool useとAIエージェント


## Function calling


In [ ]:
import json


def get_current_weather(location: str, unit: str = "celsius") -> str:
    return json.dumps({"location": location, "temperature": 20, "unit": unit})

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "Get the current weather in a given location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city, e.g. Tokyo",
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "default": "celsius",
                    },
                },
                "required": ["location"],
            },
        },
    }
]

In [ ]:
from openai import OpenAI

client = OpenAI()

messages = [
    {"role": "user", "content": "東京の現在の天気はどうですか？"},
]

response = client.chat.completions.create(
    model="gpt-5-nano",
    messages=messages,
    tools=tools,
    reasoning_effort="minimal",
)
print(response.to_json(indent=2))

In [ ]:
response_message = response.choices[0].message
messages.append(response_message.to_dict())
print(json.dumps(messages, ensure_ascii=False, indent=2))

In [ ]:
available_functions = {
    "get_current_weather": get_current_weather,
}

# 使いたい関数は複数あるかもしれないのでループ
for tool_call in response_message.tool_calls:
    # 関数を実行
    function_name = tool_call.function.name
    function_to_call = available_functions[function_name]
    function_args = json.loads(tool_call.function.arguments)
    function_response = function_to_call(**function_args)
    print(function_response.encode("utf-8").decode("unicode_escape"))

    # 関数の実行結果を会話履歴としてmessagesに追加
    messages.append(
        {
            "tool_call_id": tool_call.id,
            "role": "tool",
            "name": function_name,
            "content": function_response,
        }
    )

print(json.dumps(messages, ensure_ascii=False, indent=2))

In [ ]:
second_response = client.chat.completions.create(
    model="gpt-5-nano",
    messages=messages,
    reasoning_effort="minimal",
)
print(second_response.to_json(indent=2))